# Predicting the labour force status of young Rwandans from survey microdata

**Machine Learning summative project — African Leadership University**

| Resource | Link |
|---|---|
| GitHub repository | _add link_ |
| Google Drive (raw data zip + saved models) | _add link_ |
| Video presentation | _add link_ |

## Problem

Young people aged 16–30 are the largest group in Rwanda's working-age population, and they carry a
disproportionate share of its unemployment and labour under-utilisation. For anyone designing
career-readiness programmes it matters *which* young people end up employed, which are searching
without success, and which are outside the labour force altogether (studying, caring, discouraged
or producing only for their own household). This notebook treats that as a **three-class
classification problem** on the Rwanda Labour Force Survey (NISR): given a young person's education,
vocational training, disability, migration history, household position and unpaid domestic work,
predict their ILO labour force status.

Two modelling paradigms are compared on exactly the same data and preprocessing:

1. **Traditional machine learning** with scikit-learn (logistic regression, random forest,
   gradient boosting with XGBoost / LightGBM, and an RBF support-vector machine), with hyper-parameter search.
2. **Deep learning** with TensorFlow/Keras (multilayer perceptrons built with the Functional API,
   including entity embeddings for categorical variables), fed through `tf.data` pipelines and
   logged to TensorBoard.

Convolutional networks are not used: they exploit spatial locality in images, and survey rows have
no such structure (column order is arbitrary), so an MLP is the appropriate neural architecture.

All reusable logic is defined once in the helper cells of Section 0 (configuration, preprocessing,
evaluation, traditional and deep models) so that each experiment cell stays short and the notebook
runs on its own, locally or on Colab.

## 0. Setup

The notebook runs in two environments without edits:

* **Locally**: create the environment from `requirements.txt` (see README) and open this file from
  the `notebooks/` folder.
* **Google Colab**: the next cell mounts Google Drive, installs the two packages Colab lacks, and
  copies the Stata files from the Drive folder (`<DRIVE_PROJECT_DIR>/data/`, either the `.dta` files
  or the zipped raw dataset). Nothing else is needed: the notebook is self-contained. At the end of the notebook all outputs — the
  experiment table, figures, TensorBoard logs and saved models — are copied back to
  `<DRIVE_PROJECT_DIR>/outputs/` as evidence of the run.

Random seeds are fixed in the configuration helper and applied to NumPy, Python and TensorFlow.

In [ ]:
# --- Colab bootstrap (no effect when running locally) -----------------------------------------
import os, sys, subprocess, shutil, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_PROJECT_DIR = "ALU_ML_Summative"   # folder in My Drive holding data/ (zip or .dta files)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyreadstat", "tabulate"], check=True)
    drive_dir = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
    work = Path("/content/alu-summative-iml"); (work / "notebooks").mkdir(parents=True, exist_ok=True)
    os.chdir(work / "notebooks")
    Path("../data/raw").mkdir(parents=True, exist_ok=True)
    for z in sorted(drive_dir.glob("data/*.zip")):
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                if member.endswith(".dta") and not (Path("../data/raw") / Path(member).name).exists():
                    with zf.open(member) as src, open(Path("../data/raw") / Path(member).name, "wb") as dst:
                        shutil.copyfileobj(src, dst)
    for f in sorted(drive_dir.glob("data/*.dta")):
        if not (Path("../data/raw") / f.name).exists():
            shutil.copy(f, "../data/raw/")
    print("Colab setup complete. Raw files:", sorted(p.name for p in Path("../data/raw").glob("*.dta")))

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import tensorflow as tf, sklearn
print("TensorFlow", tf.__version__, "| scikit-learn", sklearn.__version__)

### 0.1 Helper code

Everything reusable is defined once here, in five groups, so the experiment cells below stay short.
These helpers are plain functions (no hidden package): configuration, preprocessing, evaluation,
traditional models and deep-learning models.

#### Configuration: paths, seed, population, target and feature groups

Project-wide constants: paths, seeds, target definition and feature groups.

Variable names refer to the Rwanda Labour Force Survey (RLFS) public-use
microdata produced by the National Institute of Statistics of Rwanda (NISR),
annual files 2024 and 2025 (``RW_LFS2024.dta``, ``RW_LFS2025.dta``).
See ``data/raw/RW_LFS2024_variables.tsv`` for the full variable list.

In [ ]:
from pathlib import Path

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
LOGS_DIR = ROOT / "logs"
FIGURES_DIR = ROOT / "reports" / "figures"
EXPERIMENTS_CSV = ROOT / "reports" / "experiments.csv"

RAW_GLOB = "RW_LFS*.dta"  # every annual round found in data/raw is pooled

SEED = 42

# ---------------------------------------------------------------------------
# Population and target
# ---------------------------------------------------------------------------
AGE_COL = "A04"
YOUTH_MIN_AGE, YOUTH_MAX_AGE = 16, 30  # NISR / Rwanda National Youth Policy definition ("young", YA == 1)

TARGET_COL = "status1"  # ILO labour force status, derived by NISR for persons 16+
TARGET_CODES = {1: 0, 2: 1, 3: 2}  # raw code -> class index
TARGET_NAMES = {
    0: "Employed",
    1: "Unemployed",
    2: "Outside labour force",
}
CLASS_LABELS = [TARGET_NAMES[k] for k in sorted(TARGET_NAMES)]
N_CLASSES = len(TARGET_NAMES)

# ---------------------------------------------------------------------------
# Feature groups (raw survey variables)
# ---------------------------------------------------------------------------
# Rule: only characteristics that are determined BEFORE the labour-market
# outcome may be used. Sections C (activity last 7 days, job search), D (main
# job), F (previous job / income source), H (own-use production) and every
# NISR-derived labour indicator (UR1, LFPR, TRU, PLF, LUU, neet*, IEV2, ...)
# define the target and are excluded.

NUMERIC_FEATURES = [
    "A04",    # age
    "A16Y",   # years since last move to current residence (NaN if never moved)
    "A19",    # years lived in previous location
    "B02B",   # years completed in highest level attended
    "indb1",  # years of schooling (NISR-derived from B02A/B02B)
    "hhsize", # household size (1..10+)
    "G06",    # hours last week on household chores (cooking, shopping, cleaning)
    "G07",    # hours last week caring for children / elderly
    "G01", "G02", "G03", "G04", "G05",  # hours on firewood, water, fodder, goods, construction
]

ORDINAL_FEATURES = [
    "B02A",     # highest education level attended (1 none .. 6 tertiary)
    "attained", # highest level completed (1 none .. 5 university)
    "A06", "A07", "A08", "A09", "A10", "A11",  # Washington-Group disability items (1 none .. 4 cannot)
    "TVT2",     # 1 no level completed, 2 completed general, 3 completed TVET
]

NOMINAL_FEATURES = [
    "province",
    "Code_UR",     # urban / rural
    "code_dis",    # district (30)
    "A01",         # sex
    "A02",         # relationship to household head
    "A05",         # marital status
    "A12",         # nationality
    "A13",         # residential status
    "A15",         # always lived in this district
    "B01",         # currently studying
    "B07",         # currently attending TVET school
    "B08",         # ever learned a TVET course
    "B09",         # TVET course duration (skip-pattern blank = no course)
    "B10",         # type of technical skill learned
    "B13",         # completed TVET course (1 certificate, 2 no certificate, 3 no, 4 ongoing)
    "indb05",      # field of education
    "EN01",        # household lighting energy source (asset / electrification proxy)
    "migrant",     # recent migrant: internal / international / none
    "head",        # household head flag (1 = head, 0 otherwise)
    "disable",     # NISR disability flag
    "age3_16_30",  # 16-19 / 20-24 / 25-30
    "LFS_year",    # survey round (pooled rounds)
]

# Questionnaire items that *define* the target. Used only in the leakage ablation.
LEAKY_FEATURES = ["C01", "C19"]  # paid work last 7 days; looked for work last 4 weeks

WEIGHT_COLS = ["weight2"]

ALL_RAW_COLUMNS = sorted(
    set(NUMERIC_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES + LEAKY_FEATURES + WEIGHT_COLS
        + [TARGET_COL, AGE_COL, "YA", "dG01", "dG02", "dG03", "dG04", "dG05", "dG06", "dG07", "TVT", "disable", "pid", "HHID"])
)

# Labels that mark a non-substantive answer in the NISR codebook
SENTINEL_LABEL_PATTERN = r"^(missing|refused|don.?t know|do not know|not applicable|not stated|n/?a)$"

TEST_SIZE = 0.15
VAL_SIZE = 0.15

#### Preprocessing: loading, cleaning, feature engineering, splitting, encoding

Cleaning, feature engineering and train/validation/test splitting for the RLFS.

Pipeline (deterministic given ``config.SEED``):

1. ``load_raw``            read every ``RW_LFS*.dta`` round, keep needed columns, pool
2. ``select_youth``        keep persons aged 16-30 with a valid labour force status
3. ``clean_sentinels``     Missing / Don't know codes -> NaN using the Stata value labels
4. ``engineer_features``   add interpretable composite features
5. ``split``               stratified 70/15/15 split
6. ``build_preprocessor``  sklearn ColumnTransformer fitted on train only

In [ ]:
import glob
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# ---------------------------------------------------------------------------
# 1. Loading
# ---------------------------------------------------------------------------

def raw_files(raw_dir=RAW_DIR) -> list[str]:
    return sorted(glob.glob(str(raw_dir / RAW_GLOB)))

def load_raw(raw_dir=RAW_DIR, columns: list[str] | None = None):
    '''Read and pool every RLFS round found. Returns (DataFrame with numeric codes, metadata of the last file).'''
    import pyreadstat

    columns = columns or ALL_RAW_COLUMNS
    frames, meta = [], None
    for path in raw_files(raw_dir):
        _, m = pyreadstat.read_dta(path, metadataonly=True)
        cols = [c for c in columns if c in m.column_names]
        df, meta = pyreadstat.read_dta(path, usecols=cols, apply_value_formats=False)
        year = re.search(r"(20\d{2})", path)
        df["source_file"] = path.rsplit("/", 1)[-1]
        if "LFS_year" not in df.columns or df["LFS_year"].isna().all():
            df["LFS_year"] = int(year.group(1)) if year else np.nan
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No {RAW_GLOB} file in {raw_dir}. See data/raw/README.md.")
    return pd.concat(frames, ignore_index=True, sort=False), meta

def value_labels(meta, col: str) -> dict:
    return meta.variable_value_labels.get(col, {}) if meta is not None else {}

# ---------------------------------------------------------------------------
# 2. Population and target
# ---------------------------------------------------------------------------

def select_youth(df: pd.DataFrame) -> pd.DataFrame:
    '''Keep 16-30 year-olds with a valid NISR labour force status; map status to class index.'''
    age_ok = df[AGE_COL].between(YOUTH_MIN_AGE, YOUTH_MAX_AGE)
    target_ok = df[TARGET_COL].isin(list(TARGET_CODES))
    out = df.loc[age_ok & target_ok].copy()
    out[TARGET_COL] = out[TARGET_COL].map(TARGET_CODES).astype(int)
    return out.reset_index(drop=True)

# ---------------------------------------------------------------------------
# 3. Sentinel cleaning
# ---------------------------------------------------------------------------

_SENTINEL_RE = re.compile(SENTINEL_LABEL_PATTERN, flags=re.IGNORECASE)

def sentinel_codes(meta, col: str) -> set[float]:
    return {float(v) for v, lab in value_labels(meta, col).items() if _SENTINEL_RE.match(str(lab).strip())}

def clean_sentinels(df: pd.DataFrame, meta, columns: list[str] | None = None) -> pd.DataFrame:
    out = df.copy()
    columns = columns or [c for c in out.columns if meta is not None and c in meta.column_names]
    report = {}
    for col in columns:
        codes = sentinel_codes(meta, col)
        if col == AGE_COL:
            codes |= {98.0, 99.0}
        if codes:
            mask = out[col].isin(codes)
            if mask.any():
                report[col] = int(mask.sum())
                out.loc[mask, col] = np.nan
    out.attrs["sentinel_report"] = report
    return out

# ---------------------------------------------------------------------------
# 4. Feature engineering
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    '''Add composite, interpretable features (documented inline).'''
    out = df.copy()

    # Several Stata columns arrive as strings; coerce every numeric feature.
    for col in NUMERIC_FEATURES + ["dG01", "dG02", "dG03", "dG04", "dG05", "dG06", "dG07", "head", "disable", "TVT"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # Flag variables are coded 1 = yes and blank = no (skip pattern), so blank means 0.
    for col in ["dG01", "dG02", "dG03", "dG04", "dG05", "dG06", "dG07", "head"]:
        if col in out.columns:
            out[col] = out[col].fillna(0.0)

    # Hours are only asked when the activity was done: blank hours with flag 0 mean zero hours.
    for k in range(1, 8):
        h, flag = f"G0{k}", f"dG0{k}"
        if h in out.columns and flag in out.columns:
            out[h] = out[h].where(out[h].notna() | (out[flag] != 0), 0.0)

    # care_hours: unpaid domestic + care work last week (chores + child/elder care)
    out["care_hours"] = out[["G06", "G07"]].sum(axis=1, min_count=1)
    # unpaid_work_hours: all seven own-use service activities
    out["unpaid_work_hours"] = out[[f"G0{k}" for k in range(1, 8)]].sum(axis=1, min_count=1)

    # any_disability / severe_disability from the six Washington-Group items
    wg = out[["A06", "A07", "A08", "A09", "A10", "A11"]]
    out["any_disability"] = (wg.max(axis=1) >= 2).astype(float).where(wg.notna().any(axis=1))
    out["severe_disability"] = (wg.max(axis=1) >= 3).astype(float).where(wg.notna().any(axis=1))

    # tertiary: attended tertiary education
    out["tertiary"] = (out["B02A"] == 6).astype(float).where(out["B02A"].notna())
    # tvet_completed: completed a TVET course with or without certificate
    out["tvet_completed"] = out["B13"].isin([1, 2]).astype(float).where(out["B08"].notna())
    # is_student
    out["is_student"] = (out["B01"] == 1).astype(float).where(out["B01"].notna())

    # schooling_gap: potential years of schooling (age - 6) minus years completed;
    # a large gap flags early school-leaving relative to age
    out["schooling_gap"] = (out["A04"] - 6 - out["indb1"]).clip(lower=0)

    # never_moved: A15 == 1 (always lived in this district); years since move = 0
    out["never_moved"] = (out["A15"] == 1).astype(float).where(out["A15"].notna())
    out["A16Y"] = out["A16Y"].where(out["A15"] != 1, 0.0)
    out["A19"] = out["A19"].where(out["A15"] != 1, 0.0)

    # married_female: interaction that the literature singles out for labour supply
    out["married_female"] = ((out["A01"] == 2) & out["A05"].isin([1, 2, 3])).astype(float).where(out[["A01", "A05"]].notna().all(axis=1))

    # migrant: NaN means "not a recent migrant" -> explicit code 0
    out["migrant"] = out["migrant"].fillna(0)
    # B09/B10/B13 are skip-pattern blanks for people without TVET: keep NaN (indicator) for
    # ordinal B09/B13 and an explicit category for nominal B10 (handled by the encoder).
    return out

ENGINEERED_NUMERIC = [
    "care_hours",
    "unpaid_work_hours",
    "any_disability",
    "severe_disability",
    "tertiary",
    "tvet_completed",
    "is_student",
    "schooling_gap",
    "never_moved",
    "married_female",
]
ENGINEERED_NOMINAL: list[str] = []

# ---------------------------------------------------------------------------
# 5. Splitting
# ---------------------------------------------------------------------------

@dataclass
class Split:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray

    def summary(self) -> pd.DataFrame:
        rows = {}
        for name, y in [("train", self.y_train), ("val", self.y_val), ("test", self.y_test)]:
            counts = pd.Series(y).value_counts().sort_index()
            rows[name] = {"n": len(y), **{TARGET_NAMES[k]: int(v) for k, v in counts.items()}}
        return pd.DataFrame(rows).T

def make_split(df: pd.DataFrame, feature_cols: list[str], seed: int = SEED) -> Split:
    '''Stratified 70/15/15 split on the target.'''
    X = df[feature_cols]
    y = df[TARGET_COL].to_numpy()
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    val_frac = VAL_SIZE / (1 - TEST_SIZE)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=val_frac, stratify=y_tr, random_state=seed)
    return Split(X_tr, X_va, X_te, y_tr, y_va, y_te)

# ---------------------------------------------------------------------------
# 6. Preprocessor
# ---------------------------------------------------------------------------

def feature_lists(engineered: bool = True, include_leaky: bool = False):
    numeric = NUMERIC_FEATURES + ORDINAL_FEATURES + (ENGINEERED_NUMERIC if engineered else [])
    nominal = NOMINAL_FEATURES + (ENGINEERED_NOMINAL if engineered else []) + (LEAKY_FEATURES if include_leaky else [])
    return numeric, nominal

def build_preprocessor(numeric: list[str], nominal: list[str], scale: bool = True, onehot: bool = True) -> ColumnTransformer:
    '''ColumnTransformer: median-impute (+missing indicators) and scale numerics; encode nominals.

    ``onehot=False`` yields integer codes for nominals (used for Keras embeddings).
    '''
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    if onehot:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=20)
    else:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    # Missing nominal answers (mostly skip patterns, e.g. no TVET course) become their own category.
    cat_steps = [("impute", SimpleImputer(strategy="constant", fill_value=-99)), ("encode", encoder)]
    return ColumnTransformer(
        [("num", Pipeline(num_steps), numeric), ("cat", Pipeline(cat_steps), nominal)],
        verbose_feature_names_out=False,
    )

# ---------------------------------------------------------------------------
# Convenience: full pipeline to a Split
# ---------------------------------------------------------------------------

def prepare(engineered: bool = True, include_leaky: bool = False, raw_dir=RAW_DIR):
    '''Load -> youth filter -> clean -> engineer -> split. Returns (Split, numeric, nominal, meta, youth_df).'''
    df, meta = load_raw(raw_dir)
    df = select_youth(df)
    df = clean_sentinels(df, meta)
    df = engineer_features(df)
    for col in NOMINAL_FEATURES + LEAKY_FEATURES:
        if col in df.columns:
            df[col] = df[col].astype("object")
    numeric, nominal = feature_lists(engineered, include_leaky)
    numeric = [c for c in numeric if c in df.columns]
    nominal = [c for c in nominal if c in df.columns]
    return make_split(df, numeric + nominal), numeric, nominal, meta, df

#### Evaluation: metrics, diagnostic plots and the experiment log

Metrics, diagnostic plots and the experiment log shared by every model.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import label_binarize

plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "font.size": 9})
SHORT_LABELS = ["Employed", "Unemployed", "Outside LF"]

# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def compute_metrics(y_true, y_pred, y_proba=None) -> dict:
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }
    if y_proba is not None:
        m["roc_auc_ovr_macro"] = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
        y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
        m["pr_auc_macro"] = average_precision_score(y_bin, y_proba, average="macro")
    return {k: round(float(v), 4) for k, v in m.items()}

def per_class_report(y_true, y_pred) -> pd.DataFrame:
    rep = classification_report(y_true, y_pred, target_names=CLASS_LABELS, output_dict=True, zero_division=0)
    return pd.DataFrame(rep).T.round(3)

# ---------------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------------

def plot_confusion(y_true, y_pred, title: str, path: Path | None = None, normalize: str | None = "true"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, norm, sub in zip(axes, [None, normalize], ["counts", "row-normalised"]):
        cm = confusion_matrix(y_true, y_pred, normalize=norm)
        disp = ConfusionMatrixDisplay(cm, display_labels=SHORT_LABELS)
        disp.plot(ax=ax, cmap="Blues", values_format=".2f" if norm else "d", colorbar=False)
        ax.set_title(f"{title} ({sub})")
        ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig

def plot_roc_pr(y_true, y_proba, title: str, path: Path | None = None):
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    for k in range(N_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, k], y_proba[:, k])
        ax1.plot(fpr, tpr, label=f"{SHORT_LABELS[k]} (AUC={auc(fpr, tpr):.2f})")
        prec, rec, _ = precision_recall_curve(y_bin[:, k], y_proba[:, k])
        ap = average_precision_score(y_bin[:, k], y_proba[:, k])
        ax2.plot(rec, prec, label=f"{SHORT_LABELS[k]} (AP={ap:.2f}, base={y_bin[:, k].mean():.2f})")
    ax1.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax1.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{title}: one-vs-rest ROC")
    ax2.set(xlabel="Recall", ylabel="Precision", title=f"{title}: precision-recall")
    ax1.legend(fontsize=7)
    ax2.legend(fontsize=7)
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig

def plot_sklearn_learning_curve(estimator, X, y, title: str, path: Path | None = None, cv=3, scoring="f1_macro", n_jobs=-1):
    sizes, tr, va = learning_curve(
        estimator, X, y, cv=cv, scoring=scoring, n_jobs=n_jobs,
        train_sizes=np.linspace(0.1, 1.0, 6), random_state=SEED, shuffle=True,
    )
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(sizes, tr.mean(1), "o-", label="train")
    ax.fill_between(sizes, tr.mean(1) - tr.std(1), tr.mean(1) + tr.std(1), alpha=0.15)
    ax.plot(sizes, va.mean(1), "o-", label="cross-val")
    ax.fill_between(sizes, va.mean(1) - va.std(1), va.mean(1) + va.std(1), alpha=0.15)
    ax.set(xlabel="training samples", ylabel=scoring, title=f"{title}: learning curve")
    ax.legend()
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig, {"sizes": sizes.tolist(), "train": tr.mean(1).round(4).tolist(), "val": va.mean(1).round(4).tolist()}

def plot_keras_history(history: dict, title: str, path: Path | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(history[key], label=f"train {key}")
        ax.plot(history[f"val_{key}"], label=f"val {key}")
        ax.set(xlabel="epoch", ylabel=key, title=f"{title}: {key}")
        ax.legend()
    fig.tight_layout()
    if path:
        fig.savefig(path)
    return fig

# ---------------------------------------------------------------------------
# Experiment log
# ---------------------------------------------------------------------------

def log_experiment(
    run_id: str,
    approach: str,
    model: str,
    hyperparams: dict,
    val_metrics: dict,
    test_metrics: dict | None = None,
    notes: str = "",
    features: str = "base+engineered",
    logdir: str = "",
    train_time_s: float | None = None,
    path: Path = EXPERIMENTS_CSV,
) -> pd.DataFrame:
    row = {
        "run_id": run_id,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "approach": approach,
        "model": model,
        "features": features,
        "split": "70/15/15 stratified, seed 42",
        "hyperparams": json.dumps(hyperparams, default=str),
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in (test_metrics or {}).items()},
        "train_time_s": round(train_time_s, 1) if train_time_s is not None else None,
        "logdir": logdir,
        "notes": notes,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    df = df[df.get("run_id", pd.Series(dtype=str)) != run_id] if len(df) else df
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(path, index=False)
    return df

def load_experiments(path: Path = EXPERIMENTS_CSV) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def write_tensorboard_scalars(run_id: str, metrics: dict, hyperparams: dict | None = None, logs_dir: Path = LOGS_DIR) -> str:
    '''Record final metrics (and hyper-parameters as text) for a non-Keras run so every row of the
    experiment table is backed by a TensorBoard log, not only the deep-learning runs.'''
    import tensorflow as tf

    logdir = Path(logs_dir) / run_id
    writer = tf.summary.create_file_writer(str(logdir))
    with writer.as_default():
        for k, v in metrics.items():
            tf.summary.scalar(k, float(v), step=0)
        if hyperparams:
            tf.summary.text("hyperparams", json.dumps(hyperparams, default=str), step=0)
    writer.flush()
    return str(logdir)

#### Traditional models: model zoo and a uniform fit/evaluate helper

Traditional (scikit-learn family) models and a uniform fit/evaluate helper.

In [ ]:
import time

import numpy as np
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

def model_zoo(seed: int = SEED) -> dict:
    '''Baseline configurations for each traditional algorithm.'''
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier

    return {
        "logreg": LogisticRegression(max_iter=2000, C=1.0, random_state=seed),
        "random_forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed),
        "gradient_boosting": GradientBoostingClassifier(random_state=seed),
        "xgboost": XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                 objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", device="cpu", random_state=seed, n_jobs=-1),
        "lightgbm": LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                                   random_state=seed, n_jobs=-1, verbose=-1),
        "svm_rbf": SVC(C=1.0, kernel="rbf", probability=True, random_state=seed),
    }

def fit_and_evaluate(estimator, preprocessor, split, sample_weight=None):
    '''Fit ``Pipeline(preprocessor, estimator)`` on train; return pipeline, val/test metrics, probabilities, timing.'''
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", clone(estimator))])
    t0 = time.time()
    fit_kwargs = {"model__sample_weight": sample_weight} if sample_weight is not None else {}
    pipe.fit(split.X_train, split.y_train, **fit_kwargs)
    elapsed = time.time() - t0
    out = {"pipeline": pipe, "train_time_s": elapsed}
    for name, X, y in [("val", split.X_val, split.y_val), ("test", split.X_test, split.y_test)]:
        proba = pipe.predict_proba(X)
        pred = proba.argmax(axis=1)
        out[f"{name}_metrics"] = compute_metrics(y, pred, proba)
        out[f"{name}_proba"] = proba
        out[f"{name}_pred"] = pred
    return out

def balanced_sample_weight(y: np.ndarray) -> np.ndarray:
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    w = counts.sum() / (N_CLASSES * counts)
    return w[y]

#### Deep learning: tf.data pipelines, Functional-API architectures, training loop

Keras models (Functional API), tf.data input pipelines and TensorBoard logging.

Two input modes are supported:

* ``dense``      a single preprocessed matrix (one-hot nominals), fed to an MLP
* ``embedding``  numeric block + integer-coded nominals, each nominal passed
                 through its own Embedding layer (entity embeddings)

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def set_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    keras.utils.set_random_seed(seed)

# ---------------------------------------------------------------------------
# tf.data pipelines
# ---------------------------------------------------------------------------

def make_dataset(inputs, y, batch_size: int = 256, shuffle: bool = False, seed: int = SEED) -> tf.data.Dataset:
    '''Build a batched, prefetched tf.data.Dataset from numpy inputs (array or dict of arrays).'''
    if isinstance(inputs, dict):
        feats = {k: tf.convert_to_tensor(v) for k, v in inputs.items()}
    else:
        feats = tf.convert_to_tensor(inputs, dtype=tf.float32)
    ds = tf.data.Dataset.from_tensor_slices((feats, tf.convert_to_tensor(y, dtype=tf.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 20_000), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def split_embedding_inputs(X_mat: np.ndarray, n_numeric: int, nominal_names: list[str]) -> dict:
    '''Turn a [numeric | ordinal-coded nominal] matrix into the dict the embedding model expects.'''
    out = {"numeric": X_mat[:, :n_numeric].astype(np.float32)}
    for j, name in enumerate(nominal_names):
        out[f"cat_{j}"] = (X_mat[:, n_numeric + j].astype(np.int64) + 1)  # shift so -1 (unknown) -> 0
    return out

# ---------------------------------------------------------------------------
# Architectures
# ---------------------------------------------------------------------------

def build_mlp(
    n_features: int,
    hidden: tuple[int, ...] = (256, 128),
    dropout: float = 0.3,
    batch_norm: bool = True,
    l2: float = 0.0,
    activation: str = "relu",
    name: str = "mlp",
) -> keras.Model:
    '''Plain MLP on a dense preprocessed matrix (Functional API).'''
    reg = keras.regularizers.l2(l2) if l2 else None
    inp = keras.Input(shape=(n_features,), name="features")
    x = inp
    for i, units in enumerate(hidden):
        x = layers.Dense(units, kernel_regularizer=reg, name=f"dense_{i}")(x)
        if batch_norm:
            x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.Activation(activation, name=f"act_{i}")(x)
        if dropout:
            x = layers.Dropout(dropout, name=f"drop_{i}")(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="output")(x)
    return keras.Model(inp, out, name=name)

def build_embedding_mlp(
    n_numeric: int,
    cardinalities: list[int],
    nominal_names: list[str],
    hidden: tuple[int, ...] = (256, 128),
    dropout: float = 0.3,
    batch_norm: bool = True,
    emb_dim_rule=lambda n: int(min(16, max(2, round(1.6 * n ** 0.56)))),
    l2: float = 0.0,
    name: str = "embedding_mlp",
) -> keras.Model:
    '''Entity-embedding MLP: one Embedding per nominal feature, concatenated with numerics.'''
    reg = keras.regularizers.l2(l2) if l2 else None
    num_in = keras.Input(shape=(n_numeric,), name="numeric")
    inputs, branches = [num_in], [num_in]
    for j, (card, nm) in enumerate(zip(cardinalities, nominal_names)):
        cat_in = keras.Input(shape=(1,), dtype="int64", name=f"cat_{j}")
        emb = layers.Embedding(input_dim=card + 1, output_dim=emb_dim_rule(card), name=f"emb_{nm.replace('.', '_')}")(cat_in)
        branches.append(layers.Flatten()(emb))
        inputs.append(cat_in)
    x = layers.Concatenate(name="concat")(branches)
    for i, units in enumerate(hidden):
        x = layers.Dense(units, kernel_regularizer=reg, name=f"dense_{i}")(x)
        if batch_norm:
            x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.Activation("relu", name=f"act_{i}")(x)
        if dropout:
            x = layers.Dropout(dropout, name=f"drop_{i}")(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="output")(x)
    return keras.Model(inputs, out, name=name)

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def compile_model(model: keras.Model, lr: float = 1e-3, optimizer: str = "adam", weight_decay: float = 0.0):
    opts = {"adam": keras.optimizers.Adam, "adamw": keras.optimizers.AdamW, "sgd": keras.optimizers.SGD, "rmsprop": keras.optimizers.RMSprop}
    kwargs = {"learning_rate": lr}
    if optimizer == "adamw":
        kwargs["weight_decay"] = weight_decay
    if optimizer == "sgd":
        kwargs["momentum"] = 0.9
    model.compile(optimizer=opts[optimizer](**kwargs), loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    return model

def class_weights_from(y: np.ndarray) -> dict:
    '''Inverse-frequency class weights, normalised to mean 1.'''
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    w = counts.sum() / (N_CLASSES * counts)
    return {i: float(v) for i, v in enumerate(w)}

def train(
    model: keras.Model,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    run_id: str,
    epochs: int = 100,
    patience: int = 10,
    class_weight: dict | None = None,
    reduce_lr: bool = True,
    logs_dir: Path = LOGS_DIR,
    verbose: int = 0,
):
    '''Fit with early stopping on val_loss, optional LR reduction, and TensorBoard logging.'''
    logdir = Path(logs_dir) / run_id
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.TensorBoard(log_dir=str(logdir), histogram_freq=0),
    ]
    if reduce_lr:
        callbacks.append(keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, patience // 3), min_lr=1e-6))
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, class_weight=class_weight, verbose=verbose)
    elapsed = time.time() - t0
    history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    best_epoch = int(np.argmin(history["val_loss"])) + 1
    return history, {"epochs_run": len(history["loss"]), "best_epoch": best_epoch, "train_time_s": elapsed, "logdir": str(logdir)}

def predict_proba(model: keras.Model, ds: tf.data.Dataset) -> np.ndarray:
    return model.predict(ds, verbose=0)

In [ ]:
set_seeds(SEED)
for d in [PROCESSED_DIR, MODELS_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
if EXPERIMENTS_CSV.exists():
    EXPERIMENTS_CSV.unlink()  # start the experiment table fresh on every full run
print("helpers loaded | seed", SEED, "| raw files:", [p.name for p in RAW_DIR.glob(RAW_GLOB)])

## 1. Data loading and description

**Source.** Rwanda Labour Force Survey (RLFS) public-use microdata, National Institute of Statistics
of Rwanda (NISR). The survey is a stratified two-stage probability sample of households, conducted
quarterly and pooled into annual files; it is Rwanda's official source for ILO labour-market
indicators. The annual files for 2024 (catalogue entry 114) and, when present, 2025 (entry 125) are
read from `data/raw/` and pooled. They are public-use files obtained after free registration and are
**not** committed to the repository (shared through the Google Drive folder instead).

**Why this dataset.** It is the official, nationally representative survey on exactly the question
this project asks, and it contains the pre-labour-market characteristics that career programmes can
act on: education level and field, vocational training (TVET), disability, migration, household
position and unpaid domestic work. Each annual round has roughly 100,000 persons, of whom about
27,000 are aged 16–30, so the youth sub-sample alone is above the project's 20,000-row threshold.

**Population.** Persons aged 16–30, NISR's definition of "young" (variable `YA`), with a valid
labour force status.

**Target.** `status1`, NISR's ILO-standard labour force status for persons 16+:
*Employed* (did any work for pay or profit in the last week, or temporarily absent),
*Unemployed* (not employed, actively searched in the last four weeks, and available), or
*Outside the labour force* (everyone else: students, home-makers, discouraged job-seekers,
subsistence producers).

In [ ]:
df_raw, meta = load_raw()
print("Rounds found:", sorted(df_raw["source_file"].unique()))
print("Raw rows (all ages):", f"{len(df_raw):,}", "| selected columns:", df_raw.shape[1])
ages = pd.to_numeric(df_raw["A04"], errors="coerce")
print(f"Age range {int(ages.min())}–{int(ages.max())}, median {ages.median():.0f}; share aged 16–30: {ages.between(16, 30).mean():.1%}")
display(df_raw.groupby("source_file").size().rename("persons").to_frame().T)

In [ ]:
youth = select_youth(df_raw)
print("Persons aged 16–30 with a valid labour force status:", f"{len(youth):,}")

target_counts = youth[TARGET_COL].map(TARGET_NAMES).value_counts()
fig, ax = plt.subplots(figsize=(6, 2.6))
sns.barplot(x=target_counts.values, y=target_counts.index, ax=ax, color="#4C72B0")
for i, v in enumerate(target_counts.values):
    ax.text(v + 100, i, f"{v:,} ({v/len(youth):.0%})", va="center")
ax.set(xlabel="persons", ylabel="", title="Labour force status of 16–30 year-olds (unweighted)")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig01_target_distribution.png")
plt.show()

# Survey-weighted shares for comparison with NISR's published indicators
w = pd.to_numeric(youth["weight2"], errors="coerce")
weighted = youth.groupby(youth[TARGET_COL].map(TARGET_NAMES)).apply(lambda g: w.loc[g.index].sum()) / w.sum()
display(pd.DataFrame({"unweighted share": target_counts / len(youth), "weighted share": weighted}).round(3))

About half of young Rwandans are employed, four in ten are outside the labour force and only about
one in nine is unemployed in the ILO sense. The minority class is the policy-relevant one, so
accuracy alone would reward a model that never predicts it; we report balanced accuracy, macro-F1,
per-class recall and one-vs-rest ROC-AUC throughout. The weighted shares are close to the
unweighted ones, so the unweighted sample is used for modelling and weights are kept for descriptives.

### 1.1 Exploratory analysis

How labour force status varies with the variables that career programmes can act on: sex,
residence, province, education level, TVET and current studying. Codes are mapped to codebook
labels only for plotting; the modelling pipeline keeps numeric codes.

In [ ]:
lab = lambda col: pd.to_numeric(youth[col], errors="coerce").map(meta.variable_value_labels[col])
plot_df = pd.DataFrame({
    "status": youth[TARGET_COL].map(TARGET_NAMES),
    "sex": lab("A01"), "residence": lab("Code_UR"), "province": lab("province"),
    "education": lab("B02A"), "tvet": lab("TVT"), "studying": lab("B01"),
    "age group": lab("age3_16_30"),
})
edu_order = ["None", "Pre-primary", "Primary education", "Lower secondary education", "Upper secondary education", "Tertiary education"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), ["sex", "residence", "province", "education", "tvet", "studying"]):
    ct = pd.crosstab(plot_df[col], plot_df["status"], normalize="index")
    if col == "education":
        ct = ct.reindex([e for e in edu_order if e in ct.index])
    ct.plot(kind="barh", stacked=True, ax=ax, colormap="viridis", legend=(col == "sex"))
    ax.set(title=f"Labour force status by {col}", xlabel="share", ylabel="")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig02_eda_bivariate.png")
plt.show()

In [ ]:
ct = pd.crosstab([plot_df["sex"], plot_df["age group"]], plot_df["status"], normalize="index").round(3)
ct["n"] = plot_df.groupby(["sex", "age group"]).size()
display(ct)

In [ ]:
youth_clean = clean_sentinels(youth, meta)
miss = youth_clean[NUMERIC_FEATURES + ORDINAL_FEATURES + NOMINAL_FEATURES].isna().mean().sort_values(ascending=False)
print("Raw missingness before skip-pattern handling (share of youth rows):")
display(miss.head(10).round(3).rename("share_missing").to_frame().T)

Three observations shape the preprocessing choices below. First, the raw file has almost no
"Don't know"/"Missing" codes; what looks like missingness is **skip-pattern structure**: TVET
duration and completion are blank for the 80% who never took a course, hours of chores are blank for
people who did none, and the household-head flag is blank for non-heads. Those blanks carry meaning
and are converted to zeros or explicit categories rather than imputed. Second, the share employed
rises with age while the share outside the labour force falls — the 16–19 group is dominated by
students — so age and studying will matter, and the model must not simply learn "student ⇒ outside
the labour force". Third, women and rural residents are shifted towards the outside-labour-force
class rather than towards unemployment, which is where the literature places the care-work burden.

## 2. Preprocessing and feature engineering

The preprocessing helpers are deterministic and fitted on the training split only.

1. **Leakage control (the most important choice).** NISR derives `status1` from questionnaire
   sections C (work in the last 7 days, job search in the last 4 weeks, availability), D (main job),
   F (previous job, income source) and H (own-use production). Every variable in those sections, and
   every NISR-derived labour indicator (unemployment rate, NEET, under-employment, informal
   employment, hours, earnings), *is* the target in another form. All are excluded. Only
   characteristics determined before the labour-market outcome are used. Experiment E09 adds two
   of the defining items back to show what leakage looks like.
2. **Sentinel cleaning.** Codes whose Stata value label is *Missing* or *Don't know* (e.g. 9 in the
   education items, 98 in age) are read from the file's metadata and set to NaN.
3. **Skip patterns.** Blank activity flags and household-head flags mean "no" and become 0; blank
   hours with a "did not do it" flag become 0 hours; blank migration status means "not a recent
   migrant"; blank TVET duration/skill/completion become an explicit "no TVET" category.
4. **Feature groups.** 13 numeric variables (age, years since last move, years in previous
   location, years completed in level, years of schooling, household size, seven unpaid-work hour
   counts), 9 ordered scales (education level attended and completed, six disability items, TVET
   attainment) treated as numeric, and 22 nominal variables (province, district, urban/rural, sex,
   relationship to head, marital status, nationality, residential status, always lived here,
   studying, attending TVET, learned TVET, TVET duration, TVET skill, TVET completion, field of
   education, lighting energy, migrant type, head, disability flag, age group, survey round).
5. **Engineered features** (10), each with a substantive rationale: `care_hours` (chores + care),
   `unpaid_work_hours` (all seven own-use services), `any_disability` and `severe_disability`
   (Washington Group thresholds), `tertiary`, `tvet_completed`, `is_student`,
   `schooling_gap` (potential minus completed years of schooling, a proxy for early leaving),
   `never_moved`, and `married_female` (the interaction the labour-supply literature singles out).
   Their contribution is measured directly by E01 versus E02.
6. **Encoding.** Numerics: median imputation with missing indicators, then standardisation.
   Nominals: explicit missing category, then one-hot encoding (categories with fewer than 20 training
   rows pooled) — or integer codes when feeding Keras embedding layers.
7. **Split.** Stratified 70 / 15 / 15 train / validation / test with seed 42. The validation set
   drives model selection and early stopping; the test set is touched once per final model.

In [ ]:
split, numeric, nominal, meta, youth_fe = prepare(engineered=True)
print(f"{len(numeric)} numeric/ordinal features, {len(nominal)} nominal features")
display(split.summary())

prep = build_preprocessor(numeric, nominal)
X_train_mat = prep.fit_transform(split.X_train)
print("Design matrix after one-hot encoding:", X_train_mat.shape)

display(youth_fe[ENGINEERED_NUMERIC].describe().T[["mean", "std", "min", "max"]].round(3))

In [ ]:
# Keep a processed copy for reproducibility (git-ignored)
youth_fe.to_parquet(PROCESSED_DIR / "youth_16_30_clean.parquet", index=False)
json.dump({"numeric": numeric, "nominal": nominal}, open(PROCESSED_DIR / "feature_lists.json", "w"), indent=2)